In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
import json
from pathlib import Path

os.chdir("/home/kaariaa3/mscthesis/")
sys.path.append("./src/")  # Add module directory to path

import pandas as pd

from utils.plots import calculate_metrics, calculate_accuracy
from utils.tools import get_config, collect_jobs, prettify_table

In [3]:
def score_table(jobs):
    cols = [
        "model",
        "use_instructions",
        "type_of_demonstrations",
        "number_of_demonstrations",
        "theme_accuracy",
        "topic_accuracy",
        "concept_accuracy",
        "total_accuracy",
        "theme_precision",
        "topic_precision",
        "concept_precision",
        "theme_recall",
        "topic_recall",
        "concept_recall",
        "theme_f1",
        "topic_f1",
        "concept_f1",
    ]
    table = {col_name: [] for col_name in cols}
    for job in jobs:
        df = pd.read_csv(job["result_csv"], sep=";")

        config = get_config(job["config_json"])

        # Append run configs
        table["model"].append(job["model"])
        table["use_instructions"].append(config["use_instructions"])
        table["type_of_demonstrations"].append(config["type_of_demonstrations"])
        table["number_of_demonstrations"].append(config["number_of_demonstrations"])

        # Append accuracy scores
        for name, score in calculate_accuracy(df).items():
            table[name].append(score)

        # Append precision, recall, f1 per label
        metrics = calculate_metrics(df)
        for name, score in calculate_metrics(df).items():
            table[name].append(score)

    return table

In [4]:
def aggregate_results(df):
    grouped = df.groupby(
        by=[
            "model",
            "number_of_demonstrations",
            "type_of_demonstrations",
            "use_instructions",
        ],
        as_index=True,
    )
    agg_funs = [
        "mean",
        "std",
        "count"
    ]

    agg = grouped.agg(
        {
            "theme_accuracy": agg_funs,
            "topic_accuracy": agg_funs,
            "concept_accuracy": agg_funs,
            "total_accuracy": agg_funs,
            "theme_precision": agg_funs,
            "topic_precision": agg_funs,
            "concept_precision": agg_funs,
            "theme_recall": agg_funs,
            "topic_recall": agg_funs,
            "concept_recall": agg_funs,
            "theme_f1": agg_funs,
            "topic_f1": agg_funs,
            "concept_f1": agg_funs,
        }
    )

    return agg

In [5]:
finished_jobs = collect_jobs("./outputs")
jobs_list = [job for _, job_list in finished_jobs.items() for job in job_list] # Flatten

res = pd.DataFrame(score_table(jobs_list))
res = prettify_table(res)

res_agg = aggregate_results(res)

In [7]:
res_agg[["theme_accuracy", "topic_accuracy", "concept_accuracy", "total_accuracy"]]

theme_accuracy  \
                                                                                                 mean   
model                 number_of_demonstrations type_of_demonstrations use_instructions                  
Llama-3.1-8B-Instruct 0                        mixed                  yes                    0.771963   
                      1                        negative               no                     0.613084   
                                                                      yes                    0.611838   
                                               positive               no                     0.546417   
                                                                      yes                    0.753271   
                      6                        mixed                  no                     0.768224   
                                                                      yes                    0.899688   
                                               negative               no                     0.658567   
                                                                      yes                    0.808100   
                                               positive               no                     0.657009   
                                                                      yes                    0.856698   
Qwen2.5-72B-Instruct  0                        mixed                  yes                    0.928598   
                      1                        negative               no                     0.911776   
                                                                      yes                    0.935701   
                                               positive               no                     0.885607   
                                                                      yes                    0.934953   
                      6                        mixed                  no                     0.933832   
                                                                      yes                    0.945794   
                                               negative               no                     0.926729   
                                                                      yes                    0.942523   
                                               positive               no                     0.890841   
                                                                      yes                    0.942991   
Qwen2.5-7B-Instruct   0                        mixed                  yes                    0.845234   
                      1                        negative               no                     0.697570   
                                                                      yes                    0.708411   
                                               positive               no                     0.675514   
                                                                      yes                    0.770467   
                      6                        mixed                  no                     0.699813   
                                                                      yes                    0.826916   
                                               negative               no                     0.773832   
                                                                      yes                    0.680000   
                                               positive               no                     0.614206   
                                                                      yes                    0.823551   

                                                                                                  \
                                                                                             std   
model                 number_of_demonstrations type_of_demonstrations use_instructions             
Llama-